In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# 1. パラメータ設定（lambda_12 を動かす版, 固定λを2倍にしたバージョン）
params_fixed = {
    'r_others': [1.0, 1.0, 1.0],    # [ry1, rx2, ry2]
    'lambda_diag': [1.0, 1.0],      # [lambda11, lambda22] 元は 0.5, 0.5
    'lambda_21': 0.6,               # 元は 0.3
    'c': [0.4, 0.3],                # [c1, c2]
    'd': [0.3, 0.4]                 # [d1, d2]
}

ry1, rx2, ry2 = params_fixed['r_others']
l11, l22 = params_fixed['lambda_diag']
l21 = params_fixed['lambda_21']
c1, c2 = params_fixed['c']
d1, d2 = params_fixed['d']

# 2. メッシュグリッド
res = 400
l12_vals = np.linspace(0.001, 4.0, res)  # X軸: l12（最大値を4.0に拡大）
rx1_vals = np.linspace(0.001, 3.0, res)  # Y軸: rx1
L12, RX1 = np.meshgrid(l12_vals, rx1_vals)

# 3. 計算
Det_x = c1 * d2 * l11 * l22 - c2 * d1 * L12 * l21
Det_y = l11 * l22 - L12 * l21

Num_x1 = d2 * l22 * RX1 - d1 * l21 * rx2
Num_x2 = c1 * l11 * rx2 - c2 * L12 * RX1
Num_y1 = l22 * ry1 - L12 * ry2
Num_y2 = l11 * ry2 - l21 * ry1

with np.errstate(divide='ignore', invalid='ignore'):
    X1 = Num_x1 / Det_x
    X2 = Num_x2 / Det_x
    Y1 = Num_y1 / Det_y
    Y2 = Num_y2 / Det_y

# 4. 領域のカテゴリ分け
Category = np.zeros_like(X1, dtype=int)

pos_x1 = X1 > 0
pos_x2 = X2 > 0
pos_y1 = Y1 > 0
pos_y2 = Y2 > 0

# --- Step 1: 単独条件（背景） ---
Category[~pos_y1] = 2
Category[~pos_y2] = 3
Category[~pos_x1] = 4
Category[~pos_x2] = 5

# --- Step 2: 複合条件（優先度高） ---
mask_green = (~pos_y2) & (~pos_x2)
mask_gray  = (~pos_y1) & (~pos_x1)
mask_red   = (~pos_y2) & (~pos_x1)
mask_blue  = (~pos_y1) & (~pos_x2)

Category[mask_green] = 6
Category[mask_gray]  = 7
Category[mask_red]   = 8
Category[mask_blue]  = 9

# --- Step 3: 共存（最優先） ---
mask_coexist = pos_x1 & pos_x2 & pos_y1 & pos_y2
Category[mask_coexist] = 1

# 5. プロット
fig, ax = plt.subplots(figsize=(12, 8))

colors = [
    'white',          # 0
    'gold',           # 1: Coexist
    'skyblue',        # 2: y1<0
    'salmon',         # 3: y2<0
    'SlateBlue',      # 4: x1<0
    'IndianRed',      # 5: x2<0
    'MediumSeaGreen', # 6: Green
    'lightgray',      # 7: Gray
    'red',            # 8: Red
    'blue'            # 9: Blue
]

cmap = ListedColormap(colors)
norm = BoundaryNorm(np.arange(-0.5, 10.5, 1), cmap.N)

im = ax.imshow(
    Category,
    extent=[0, 4.0, 0, 3.0],  # X軸の最大値を4.0に
    origin='lower',
    cmap=cmap,
    norm=norm,
    aspect='auto',
    alpha=0.9,
)

# --- 補助線 ---
# Det_y = 0 line (L12について解く)
singularity_y_val = (l11 * l22) / l21

# Det_x = 0 line (L12について解く)
num_sing_x = c1 * d2 * l11 * l22
den_sing_x = c2 * d1 * l21
singularity_x_val = num_sing_x / den_sing_x

# Num_x2 = 0 boundary: RX1 = (c1*l11*rx2)/(c2*L12)
x_curve = np.linspace(0.001, 4.0, 1000)
y_curve = (c1 * l11 * rx2) / (c2 * x_curve)

ax.axvline(x=singularity_y_val, color='white', linestyle='-', linewidth=3)
ax.axvline(x=singularity_x_val, color='black', linestyle='--', linewidth=3)
ax.plot(x_curve, y_curve, color='purple', linestyle=':', linewidth=2)

# --- ラベル設定 ---
ax.set_xlabel(r'$\mathrm{Interspecific\ Competition}\ \lambda_{12}$')
ax.set_ylabel(r'$\mathrm{Intrinsic\ Growth\ Rate}\ r_{x1}$')
ax.set_title(r'$\mathrm{Phase\ Diagram}\ (\lambda_{12}\ \mathrm{vs}\ r_{x1})$')
ax.set_xlim(0, 4.0)
ax.set_ylim(0, 3.0)

textstr = '\n'.join((
    r'Fixed Parameters:',
    r'$\lambda_{11} = %.1f, \lambda_{22} = %.1f$' % (l11, l22),
    r'$\lambda_{21} = %.1f$' % l21,
    r'$r_{others} = [%.1f, %.1f, %.1f]$' % (ry1, rx2, ry2),
    r'$c=[%.1f, %.1f], d=[%.1f, %.1f]$' % (c1, c2, d1, d2)
))
props = dict(boxstyle='round', facecolor='white', alpha=0.9)
ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=10, verticalalignment='top', bbox=props)

legend_elements = [
    Patch(facecolor='gold', edgecolor='k', label=r'$\mathrm{Coexistence}$'),
    Patch(facecolor='MediumSeaGreen', edgecolor='k', label=r'$y_2 < 0\ \mathrm{and}\ x_2 < 0\ \mathrm{(Green)}$'),
    Patch(facecolor='lightgray', edgecolor='k', label=r'$y_1 < 0\ \mathrm{and}\ x_1 < 0\ \mathrm{(Gray)}$'),
    Patch(facecolor='red', edgecolor='k', label=r'$y_2 < 0\ \mathrm{and}\ x_1 < 0\ \mathrm{(Red)}$'),
    Patch(facecolor='blue', edgecolor='k', label=r'$y_1 < 0\ \mathrm{and}\ x_2 < 0\ \mathrm{(Blue)}$'),
    Patch(facecolor='skyblue', edgecolor='k', label=r'$\mathrm{Only}\ y_1 < 0$'),
    Patch(facecolor='salmon', edgecolor='k', label=r'$\mathrm{Only}\ y_2 < 0$'),
    Patch(facecolor='SlateBlue', edgecolor='k', label=r'$\mathrm{Only}\ x_1 < 0$'),
    Patch(facecolor='IndianRed', edgecolor='k', label=r'$\mathrm{Only}\ x_2 < 0$'),
    Line2D([0], [0], color='white', lw=3, label=r'$Det_y=0$'),
    Line2D([0], [0], color='black', linestyle='--', lw=3, label=r'$Det_x=0$'),
    Line2D([0], [0], color='purple', linestyle=':', lw=2, label=r'$Num_{x2}=0$')
]

plt.subplots_adjust(right=0.7)
ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.05, 1.0), borderaxespad=0.)
# 出力ディレクトリ設定
output_dir = Path("../phase_diagram_images")
output_dir.mkdir(exist_ok=True, parents=True)

# 画像を保存
output_path = output_dir / "l12_rx1_lambda2x.png"
fig.savefig(output_path, dpi=300, bbox_inches="tight")
plt.close(fig)

plt.show()